## How is this different from TIS?

Earlier TIS was:

$$w_t = \min\left( \frac{\pi_{\text{old}}}{\pi_{\text{rollout}}},\, C \right)$$

That is **upper truncation only**.

Example:

```
ratio = 0.1 -> keep 0.1
ratio = 1.0 -> keep 1.0
ratio = 5.0 -> cap to C
```

IcePop is different. It is **double-sided masking**:

```
if ratio too small -> discard token
if ratio healthy   -> keep token, weighted by ratio
if ratio too large -> discard token
```

Example:

```
alpha = 0.8
beta  = 1.2

ratio = 0.50 -> discard
ratio = 0.95 -> keep
ratio = 1.10 -> keep
ratio = 2.00 -> discard
```

So IcePop is **stricter** than TIS.

# ICE pop

$$
\small{
\begin{aligned}
\mathcal{J}_{\text{IcePop}}(\theta) = -\mathbb{E}_{x \sim \mathcal{D},\, \{y_i\}_{i=1}^G \sim \pi_{\textcolor{red}{\text{infer}}}(\cdot \mid x;\, \theta_{\text{old}})} \Bigg[ &\frac{1}{G} \sum_{i=1}^G \frac{1}{|y_i|} \sum_{t=1}^{|y_i|} \bigg[ \mathcal{M}\Big(\frac{\pi_{\textcolor{blue}{\text{train}}}(y_{i,t} \mid x, y_{i,<t};\, \theta_{\text{old}})}{\pi_{\textcolor{red}{\text{infer}}}(y_{i,t} \mid x, y_{i,<t};\, \theta_{\text{old}})};\, \alpha, \beta\Big) \\
&\cdot \min\Big( r_{i,t}\, \textcolor{purple}{\widehat{A}_{i,t}},\, \text{clip}(r_{i,t},\, 1 - \varepsilon,\, 1 + \varepsilon)\, \textcolor{purple}{\widehat{A}_{i,t}} \Big) \bigg] \Bigg]
\end{aligned}
}
$$

That $\mathcal{M}(\cdot)$ is the **masking function**.

The equation does not write the word "mask" directly in English, but mathematically the masking is done by $\mathcal{M}$.

The paper defines it like:

$$
\mathcal{M}(k) = \begin{cases} k, & \text{if } k \in [\alpha,\beta] \\ 0, & \text{otherwise} \end{cases}
$$

where:

$$
k = \frac{\pi_{\text{train}}(y_{i,t} \mid x, y_{i,<t};\, \theta_{\text{old}})}{\pi_{\text{infer}}(y_{i,t} \mid x, y_{i,<t};\, \theta_{\text{old}})}
$$

So the masking happens here:

$$
\mathcal{M}(k;\, \alpha, \beta) \cdot \min\Big( r_{i,t}\, \widehat{A}_{i,t},\, \text{clip}(r_{i,t},\, 1-\epsilon,\, 1+\epsilon)\, \widehat{A}_{i,t} \Big)
$$

In [ ]:
# Double-sided train/inference mismatch masking
# ============================================================

import copy
import numpy as np
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# 0) Load student and teacher
# ------------------------------------------------------------

base_policy = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Current trainable student policy: πθ
student_policy = copy.deepcopy(base_policy).to(device)

# Keep dropout off for demo clarity.
# Gradients still flow because requires_grad=True.
student_policy.eval()
for p in student_policy.parameters():
    p.requires_grad_(True)

# Frozen teacher policy: ν
# In real OPD, this is usually a stronger / larger model.
teacher_policy = copy.deepcopy(base_policy).to(device).eval()
for p in teacher_policy.parameters():
    p.requires_grad_(False)


In [ ]:
# ------------------------------------------------------------
# 1) Teacher-forced prompt + completion encoding
# ------------------------------------------------------------

def encode_prompt_response(tokenizer, prompt_text: str, response_text: str):
    """
    Teacher-forced prompt + response encoding.

    Returns:
        input_ids:    [1, seq_len]
        prompt_len:   number of prompt tokens
        response_len: number of response tokens
    """
    prompt_ids = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"]  # [1, prompt_len]

    full_ids = tokenizer(
        prompt_text + response_text,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"]  # [1, full_len]

    prompt_len = prompt_ids.shape[1]
    response_len = full_ids.shape[1] - prompt_len
    
    assert response_len > 0, "Response must add at least one token."

    return full_ids, prompt_len, response_len

In [ ]:
# ------------------------------------------------------------
# 2) Selected-token log-probs for response tokens
# ------------------------------------------------------------

def selected_logprobs_for_response(
    model,
    tokenizer,
    prompt: str,
    completion: str,
    require_grad: bool,
    temperature: float = 1.0,
):
    """
    Teacher-forced scoring of completion tokens.

    For completion token at absolute position pos,
    probability comes from logits at pos - 1.

    Returns:
        response_token_ids: [T]
        selected_logps:     [T]
    """
    input_ids, prompt_len, _ = encode_prompt_response(
        tokenizer,
        prompt,
        completion,
    )

    model_device = next(model.parameters()).device
    input_ids = input_ids.to(model_device)

    ctx = torch.enable_grad() if require_grad else torch.no_grad()

    with ctx:
        out = model(input_ids, use_cache=False)

        logits = out.logits / temperature
        logp_all = F.log_softmax(logits, dim=-1)

        token_ids = input_ids[0]  # [seq_len]
        seq_len = token_ids.shape[0]

        response_token_ids = token_ids[prompt_len:]  # [T]

        # response token at pos is predicted by logits at pos - 1
        prev_pos = torch.arange(
            prompt_len - 1,
            seq_len - 1,
            device=model_device,
        )  # [T]

        lp_rows = logp_all[0, prev_pos, :]  # [T, vocab]

        selected_logps = lp_rows.gather(
            dim=1,
            index=response_token_ids.unsqueeze(1),
        ).squeeze(1)  # [T]

    return response_token_ids, selected_logps


In [ ]:
def icepop_mismatch_weight(
    logp_train_old: torch.Tensor,
    logp_infer_old: torch.Tensor,
    alpha: float,
    beta: float,
):
    """
    IcePop masking:

        k = pi_train_old / pi_infer_old

        M(k; alpha, beta) =
            k, if alpha <= k <= beta
            0, otherwise

    Returns:
        weight:    [T], M(k)
        keep_frac: scalar, fraction of tokens kept
    """
    assert alpha > 0.0
    assert beta > alpha

    log_k = logp_train_old - logp_infer_old

    log_alpha = torch.log(
        torch.tensor(alpha, device=log_k.device, dtype=log_k.dtype)
    )

    log_beta = torch.log(
        torch.tensor(beta, device=log_k.device, dtype=log_k.dtype)
    )

    keep_mask = (log_k >= log_alpha) & (log_k <= log_beta)

    k = torch.exp(log_k)

    # you can also use k.masked_fill(~keep_mask, 0.0) and then return k.detach() directly as weight
    weight = torch.where(
        keep_mask,
        k,
        torch.zeros_like(k),
    )

    keep_frac = keep_mask.float().mean()

    return weight.detach(), keep_frac.detach()

In [ ]:
# ------------------------------------------------------------
# 4) Collect one teacher-forced IcePop OPD item
# ------------------------------------------------------------

@torch.no_grad()
def collect_one_teacher_forced_icepop_opd_item(
    infer_policy,
    train_old_policy,
    teacher_policy,
    tokenizer,
    prompt: str,
    completion: str,
    teacher_temperature: float,
    infer_scoring_temperature: float,
    icepop_alpha: float,
    icepop_beta: float,
):
    """
    Collect fixed rollout item using teacher forcing.

    We treat `completion` as already sampled from π_infer.

    Roles:
        π_infer:
            inference/sampler policy that generated/scored rollout tokens

        π_train_old:
            old training-backend learner policy

        ν:
            frozen teacher policy

    Stored:
        logp_infer_old
        logp_train_old
        logp_teacher
        teacher_signal = sg(logp_teacher - logp_train_old)
        icepop_weight = M(π_train_old / π_infer_old; alpha, beta)
    """
    ids_infer, logp_infer_old = selected_logprobs_for_response(
        infer_policy,
        tokenizer,
        prompt,
        completion,
        require_grad=False,
        temperature=infer_scoring_temperature,
    )

    ids_train_old, logp_train_old = selected_logprobs_for_response(
        train_old_policy,
        tokenizer,
        prompt,
        completion,
        require_grad=False,
        temperature=1.0,
    )

    ids_teacher, logp_teacher = selected_logprobs_for_response(
        teacher_policy,
        tokenizer,
        prompt,
        completion,
        require_grad=False,
        temperature=teacher_temperature,
    )

    assert torch.equal(ids_infer, ids_train_old), (
        "Token mismatch between inference policy and train-old policy."
    )
    assert torch.equal(ids_infer, ids_teacher), (
        "Token mismatch between inference policy and teacher policy."
    )

    icepop_weight, icepop_keep_frac = icepop_mismatch_weight(
        logp_train_old=logp_train_old,
        logp_infer_old=logp_infer_old,
        alpha=icepop_alpha,
        beta=icepop_beta,
    )

    # OPD teacher signal anchored at train-old policy.
    # This is frozen at rollout-collection time.
    teacher_signal = (logp_teacher - logp_train_old).detach()

    return {
        "prompt": prompt,
        "completion": completion,
        "response_token_ids": ids_infer.detach().cpu(),
        "logp_infer_old": logp_infer_old.detach().cpu(),
        "logp_train_old": logp_train_old.detach().cpu(),
        "logp_teacher": logp_teacher.detach().cpu(),
        "teacher_signal": teacher_signal.detach().cpu(),
        "icepop_weight": icepop_weight.detach().cpu(),
        "icepop_keep_frac": float(icepop_keep_frac.cpu()),
    }



In [ ]:
# ------------------------------------------------------------
# 5) Collect teacher-forced IcePop OPD batch/list
# ------------------------------------------------------------

@torch.no_grad()
def collect_teacher_forced_icepop_opd_batch(
    infer_policy,
    train_old_policy,
    teacher_policy,
    tokenizer,
    prompt: str,
    completions: list[str],
    teacher_temperature: float,
    infer_scoring_temperature: float,
    icepop_alpha: float,
    icepop_beta: float,
):
    """
    Collect multiple fixed completions for the same prompt.

    Aggregation later:
        mean over tokens within each completion,
        then mean over completions.
    """
    rollout_items = []

    for completion in completions:
        item = collect_one_teacher_forced_icepop_opd_item(
            infer_policy=infer_policy,
            train_old_policy=train_old_policy,
            teacher_policy=teacher_policy,
            tokenizer=tokenizer,
            prompt=prompt,
            completion=completion,
            teacher_temperature=teacher_temperature,
            infer_scoring_temperature=infer_scoring_temperature,
            icepop_alpha=icepop_alpha,
            icepop_beta=icepop_beta,
        )

        rollout_items.append(item)

    return rollout_items

In [ ]:
# ------------------------------------------------------------
# 6) Per-sequence IcePop OPD objective
# ------------------------------------------------------------

def icepop_opd_objective_for_one_sequence(
    student_policy,
    tokenizer,
    rollout_item: dict,
    clip_eps: float,
    require_grad: bool,
):
    """
    Compute one sequence's IcePop OPD objective.

    For one fixed completion:

        IcePop mismatch weight:
            M_t = M(π_train_old / π_infer_old; alpha, beta)

        PPO update ratio:
            r_t(θ) = πθ(y_t | s_t) / π_train_old(y_t | s_t)

        Frozen OPD teacher signal:
            A_t = sg(log ν(y_t | s_t) - log π_train_old(y_t | s_t))

        PPO-style clipped objective:
            min(r_t * A_t, clip(r_t, 1-eps, 1+eps) * A_t)

        IcePop objective:
            M_t * min(...)

        Sequence objective:
            mean over tokens in this completion
    """
    prompt = rollout_item["prompt"]
    completion = rollout_item["completion"]

    token_ids_old = rollout_item["response_token_ids"].to(device)
    logp_train_old = rollout_item["logp_train_old"].to(device)
    teacher_signal = rollout_item["teacher_signal"].to(device)
    icepop_weight = rollout_item["icepop_weight"].to(device)

    ids_current, logp_current = selected_logprobs_for_response(
        student_policy,
        tokenizer,
        prompt,
        completion,
        require_grad=require_grad,
        temperature=1.0,
    )

    assert torch.equal(ids_current, token_ids_old), (
        "Current student token mismatch with rollout batch."
    )

    # PPO-style current/train-old ratio:
    ratio = torch.exp(logp_current - logp_train_old)

    ratio_clipped = torch.clamp(
        ratio,
        min=1.0 - clip_eps,
        max=1.0 + clip_eps,
    )

    surr1 = ratio * teacher_signal
    surr2 = ratio_clipped * teacher_signal

    surrogate = torch.minimum(surr1, surr2)

    # IcePop double-sided masking:
    weighted_surrogate = icepop_weight * surrogate

    # Correct OPD aggregation for one sequence:
    sequence_objective = weighted_surrogate.mean()

    with torch.no_grad():
        clipfrac = (
            ((ratio > 1.0 + clip_eps) | (ratio < 1.0 - clip_eps))
            .float()
            .mean()
            .item()
        )

        nonzero_icepop_frac = (icepop_weight > 0).float().mean().item()

        seq_stats = {
            "sequence_objective": float(sequence_objective.detach().cpu()),
            "response_len": int(logp_current.numel()),
            "ratio_mean": float(ratio.detach().mean().cpu()),
            "ratio_min": float(ratio.detach().min().cpu()),
            "ratio_max": float(ratio.detach().max().cpu()),
            "clipfrac": clipfrac,
            "teacher_signal_mean": float(teacher_signal.mean().cpu()),
            "teacher_signal_min": float(teacher_signal.min().cpu()),
            "teacher_signal_max": float(teacher_signal.max().cpu()),
            "icepop_weight_mean": float(icepop_weight.mean().cpu()),
            "icepop_weight_min": float(icepop_weight.min().cpu()),
            "icepop_weight_max": float(icepop_weight.max().cpu()),
            "icepop_keep_frac": rollout_item["icepop_keep_frac"],
            "nonzero_icepop_frac": nonzero_icepop_frac,
        }

    return sequence_objective, seq_stats


In [ ]:
# ------------------------------------------------------------
# 7) Batch-level objective: seq-mean-token-mean
# ------------------------------------------------------------

def icepop_opd_batch_objective_seq_mean_token_mean(
    student_policy,
    tokenizer,
    rollout_items: list[dict],
    clip_eps: float,
    require_grad: bool,
):
    """
    Correct aggregation:

        objective =
            mean_i [
                mean_t objective_{i,t}
            ]

    This gives each completion equal weight,
    regardless of completion length.
    """
    sequence_objectives = []
    sequence_stats = []

    for item in rollout_items:
        seq_obj, seq_stat = icepop_opd_objective_for_one_sequence(
            student_policy=student_policy,
            tokenizer=tokenizer,
            rollout_item=item,
            clip_eps=clip_eps,
            require_grad=require_grad,
        )

        sequence_objectives.append(seq_obj)
        sequence_stats.append(seq_stat)

    objective = torch.stack(sequence_objectives).mean()
    loss = -objective

    with torch.no_grad():
        response_lens = [s["response_len"] for s in sequence_stats]
        clipfracs = [s["clipfrac"] for s in sequence_stats]
        ratio_means = [s["ratio_mean"] for s in sequence_stats]
        icepop_weight_means = [s["icepop_weight_mean"] for s in sequence_stats]
        icepop_keep_fracs = [s["icepop_keep_frac"] for s in sequence_stats]
        teacher_signal_means = [s["teacher_signal_mean"] for s in sequence_stats]

        batch_stats = {
            "loss": float(loss.detach().cpu()),
            "objective": float(objective.detach().cpu()),
            "num_sequences": len(rollout_items),
            "response_lens": response_lens,
            "mean_response_len": float(np.mean(response_lens)),
            "mean_clipfrac": float(np.mean(clipfracs)),
            "mean_ratio": float(np.mean(ratio_means)),
            "mean_icepop_weight": float(np.mean(icepop_weight_means)),
            "mean_icepop_keep_frac": float(np.mean(icepop_keep_fracs)),
            "mean_teacher_signal": float(np.mean(teacher_signal_means)),
            "per_sequence_objective": [
                s["sequence_objective"] for s in sequence_stats
            ],
            "per_sequence_stats": sequence_stats,
        }

    return loss, batch_stats


In [ ]:
# ------------------------------------------------------------
# 8) IcePop OPD update step
# ------------------------------------------------------------

def icepop_opd_update_step_teacher_forced(
    student_policy,
    optimizer,
    tokenizer,
    rollout_items: list[dict],
    clip_eps: float,
):
    """
    One actor update on fixed teacher-forced IcePop OPD rollout items.

    Current student πθ recomputes log-probs with gradients.
    Rollout tensors are fixed/detached.
    """
    loss, stats = icepop_opd_batch_objective_seq_mean_token_mean(
        student_policy=student_policy,
        tokenizer=tokenizer,
        rollout_items=rollout_items,
        clip_eps=clip_eps,
        require_grad=True,
    )

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    return stats

In [ ]:
# ------------------------------------------------------------
# 9) Eval helper
# ------------------------------------------------------------

@torch.no_grad()
def icepop_opd_eval_stats_teacher_forced(
    student_policy,
    tokenizer,
    rollout_items: list[dict],
    clip_eps: float,
):
    """
    Compute the same stats without updating.
    """
    loss, stats = icepop_opd_batch_objective_seq_mean_token_mean(
        student_policy=student_policy,
        tokenizer=tokenizer,
        rollout_items=rollout_items,
        clip_eps=clip_eps,
        require_grad=False,
    )

    return stats

In [ ]:
# ------------------------------------------------------------
# 10) Settings
# ------------------------------------------------------------

# Actor PPO epochs over one fixed sampled/rollout batch.
# This mirrors the PPO-style actor update infrastructure.
actor_ppo_epochs = 2

# Teacher temperature is only for this toy demo because teacher and student
# start from the same tiny checkpoint. In real OPD, teacher is usually different.
teacher_temperature = 0.7

# In a real MoE/vLLM setup, π_infer and π_train_old can differ naturally.
# In this HF-only toy demo, they are identical if temperature=1.0.
#
# Keep 1.0 for faithful scoring.
# Set e.g. 1.2 only if you want to artificially simulate mismatch.
infer_scoring_temperature = 1.0

# IcePop double-sided mask interval.
# Keep tokens only when:
#     alpha <= π_train_old / π_infer_old <= beta
icepop_alpha = 0.8
icepop_beta = 1.25

# PPO-style current/train-old clipping.
clip_eps = 0.2

lr = 1e-5
optimizer = torch.optim.Adam(student_policy.parameters(), lr=lr)


# ------------------------------------------------------------
# 11) Fixed teacher-forced examples
# ------------------------------------------------------------

examples = [
    {
        "prompt": "The capital of France is",
        "completions": [
            " Paris.",
            " London.",
        ],
    },
    {
        "prompt": "2 + 2 =",
        "completions": [
            " 4.",
            " 5.",
        ],
    },
]




# ------------------------------------------------------------
# 12) Main teacher-forced IcePop OPD flow
# ------------------------------------------------------------

for example_idx, ex in enumerate(examples, start=1):

    print("\n" + "=" * 80)
    print(f"TEACHER-FORCED ICEPOP OPD EXAMPLE {example_idx}")
    print("=" * 80)

    prompt = ex["prompt"]
    completions = ex["completions"]

    print("Prompt:", repr(prompt))
    print("Completions:")
    for c in completions:
        print("   ", repr(c))

    # --------------------------------------------------------
    # A) Create policy roles
    # --------------------------------------------------------
    # π_infer: behavior/inference/sampler policy
    infer_policy = copy.deepcopy(student_policy).to(device).eval()
    for p in infer_policy.parameters():
        p.requires_grad_(False)

    # π_train_old: old training-backend learner policy
    train_old_policy = copy.deepcopy(student_policy).to(device).eval()
    for p in train_old_policy.parameters():
        p.requires_grad_(False)

    print("\n[1] Policy roles")
    print("    π_infer:      frozen inference/sampler policy")
    print("    π_train_old:  frozen old training-backend policy")
    print("    πθ:           current trainable student_policy")
    print("    ν:            frozen teacher_policy")
    print("    In this HF demo, π_infer and π_train_old are identical unless you")
    print("    intentionally change infer_scoring_temperature to simulate mismatch.")

    # --------------------------------------------------------
    # B) Collect teacher-forced rollout items once
    # --------------------------------------------------------
    print("\n[2] Collect fixed IcePop OPD rollout batch once")

    rollout_items = collect_teacher_forced_icepop_opd_batch(
        infer_policy=infer_policy,
        train_old_policy=train_old_policy,
        teacher_policy=teacher_policy,
        tokenizer=tokenizer,
        prompt=prompt,
        completions=completions,
        teacher_temperature=teacher_temperature,
        infer_scoring_temperature=infer_scoring_temperature,
        icepop_alpha=icepop_alpha,
        icepop_beta=icepop_beta,
    )

    for i, item in enumerate(rollout_items):
        print(f"    item {i}:")
        print("        completion:", repr(item["completion"]))
        print("        response_len:", int(item["response_token_ids"].numel()))
        print("        teacher_signal mean/min/max:",
              float(item["teacher_signal"].mean()),
              float(item["teacher_signal"].min()),
              float(item["teacher_signal"].max()))
        print("        icepop_weight mean/min/max:",
              float(item["icepop_weight"].mean()),
              float(item["icepop_weight"].min()),
              float(item["icepop_weight"].max()))
        print("        icepop_keep_frac:",
              float(item["icepop_keep_frac"]))

    # --------------------------------------------------------
    # C) Before actor update
    # --------------------------------------------------------
    print("\n[3] Before IcePop OPD update")

    before = icepop_opd_eval_stats_teacher_forced(
        student_policy=student_policy,
        tokenizer=tokenizer,
        rollout_items=rollout_items,
        clip_eps=clip_eps,
    )

    print("    loss:", before["loss"])
    print("    objective:", before["objective"])
    print("    num_sequences:", before["num_sequences"])
    print("    response_lens:", before["response_lens"])
    print("    mean_response_len:", before["mean_response_len"])
    print("    mean_ratio:", before["mean_ratio"])
    print("    mean_icepop_weight:", before["mean_icepop_weight"])
    print("    mean_icepop_keep_frac:", before["mean_icepop_keep_frac"])
    print("    mean_teacher_signal:", before["mean_teacher_signal"])
    print("    mean_clipfrac:", before["mean_clipfrac"])
    print("    per_sequence_objective:", before["per_sequence_objective"])

    # --------------------------------------------------------
    # D) Actor PPO epochs over same fixed rollout batch
    # --------------------------------------------------------
    print("\n[4] IcePop OPD actor update loop")
    print("    actor_ppo_epochs:", actor_ppo_epochs)
    print("    Fixed rollout batch is reused across actor epochs.")
    print("    π_train_old logprobs stay fixed.")
    print("    teacher_signal = sg(log ν - log π_train_old) stays fixed.")
    print("    icepop_weight = M(π_train_old / π_infer; alpha, beta) stays fixed.")
    print("    πθ logprobs are recomputed each epoch and updated.")

    for epoch in range(1, actor_ppo_epochs + 1):

        stats = icepop_opd_update_step_teacher_forced(
            student_policy=student_policy,
            optimizer=optimizer,
            tokenizer=tokenizer,
            rollout_items=rollout_items,
            clip_eps=clip_eps,
        )

        print(f"\n    actor PPO epoch {epoch}")
        print("        loss:", stats["loss"])
        print("        objective:", stats["objective"])
        print("        mean_ratio:", stats["mean_ratio"])
        print("        mean_icepop_weight:", stats["mean_icepop_weight"])
        print("        mean_icepop_keep_frac:", stats["mean_icepop_keep_frac"])
        print("        mean_teacher_signal:", stats["mean_teacher_signal"])
        print("        mean_clipfrac:", stats["mean_clipfrac"])
        print("        per_sequence_objective:", stats["per_sequence_objective"])

    # --------------------------------------------------------
    # E) After update
    # --------------------------------------------------------
    print("\n[5] After IcePop OPD update on same fixed rollout batch")

    after = icepop_opd_eval_stats_teacher_forced(
        student_policy=student_policy,
        tokenizer=tokenizer,
        rollout_items=rollout_items,
        clip_eps=clip_eps,
    )

    print("    loss:", after["loss"])
    print("    objective:", after["objective"])
    print("    mean_ratio:", after["mean_ratio"])
    print("    mean_icepop_weight:", after["mean_icepop_weight"])
    print("    mean_icepop_keep_frac:", after["mean_icepop_keep_frac"])
    print("    mean_teacher_signal:", after["mean_teacher_signal"])
    print("    mean_clipfrac:", after["mean_clipfrac"])
    print("    per_sequence_objective:", after["per_sequence_objective"])

    print("\n[6] Discard this fixed rollout batch")
    print("    Next prompt creates fresh π_infer and π_train_old from updated student.")